# Final Assembly: Institutional Backtest Execution

This notebook consolidates all architectural components (`DataHandler`, `Strategy`, `FrictionModel`, `Portfolio`) and executes them through the `BacktestEngine`. It includes the mathematical rigor required for undestanding the documentation with visualizations.

## 1. Theoretical Foundations (Mathematical Rigor)

The following mathematical definitions govern the behavior of the algorithm, position sizing, and portfolio performance evaluation. All operations in the backtesting logic have been strictly vectorized using `pandas` and `numpy` to guarantee an asymptotic array complexity of $\mathcal{O}(1)$.

### 1.1 Simple Moving Average (SMA)
The Simple Moving Average acts as a low-pass filter to smooth out high-frequency noise in the price time series. It is defined as the unweighted arithmetic mean of closing prices over the past $n$ periods:
$$ SMA_t = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i} $$
Where:
- $P_t$: The closing price of the asset at time $t$.
- $n$: The lookback window length.

**Obs**: By comparing a short-term SMA with a long-term SMA, we can mathematically capture momentum shifts when the fast-moving average crosses the slow-moving average.

### 1.2 Momentum
Standardized momentum relies on the sign of the historical return over a specified lookback period $n$. It assumes that assets exhibiting positive recent returns will continue to perform well.
$$ M_t = P_t - P_{t-n} $$
$$ Signal_t = \text{sign}(M_t) $$
Where:
- $M_t$: The absolute momentum at time $t$.
- $Signal_t \in \{-1, 0, 1\}$: The trading signal, instructing a short, neutral (flat), or long position respectively.

### 1.3 Sharpe Ratio
The Sharpe Ratio measures the performance of an investment adjusted for its risk. It is the expected excess return per unit of volatility.
$$ SR = \frac{\mathbb{E}[R_p] - R_f}{\sigma_p} \sqrt{252} $$
Where:
- $\mathbb{E}[R_p]$: The expected return of the portfolio.
- $R_f$: The risk-free rate of return (assumed $0$ for simplicity in this baseline).
- $\sigma_p$: The standard deviation of the portfolio returns (volatility).
- $\sqrt{252}$: Annualization factor assuming 252 trading days in a year.

### 1.4 Maximum Drawdown (MDD)
Maximum Drawdown evaluates the worst-case historical loss from a peak to a subsequent trough before a new peak is achieved. It is a critical metric for tail-risk management.
$$ MDD = \max_{t \in (0, T)} \left( \frac{P_{peak} - P_t}{P_{peak}} \right) $$
$$ \text{where } P_{peak} = \max_{\tau \in (0, t)} P_\tau $$
Where:
- $P_t$: Portfolio equity at time $t$.
- $P_{peak}$: The rolling maximum equity achieved up to time $t$.

### 1.5 Compound Annual Growth Rate (CAGR)
CAGR provides the smoothed annualized geometric growth rate of the portfolio, ignoring intra-year volatility.
$$ CAGR = \left( \frac{EV}{BV} \right)^{\frac{1}{y}} - 1 $$
Where:
- $EV$: Ending Value of the portfolio.
- $BV$: Beginning Value of the portfolio.
- $y$: The number of years the investment was held ($n / 252$).

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Add the root directory to the path to allow modular import of src
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src.infrastructure.ParquetDataHandler import ParquetDataHandler
from src.core.SMAStrategy import SMAStrategy
from src.core.ProportionalFrictionModel import ProportionalFrictionModel
from src.core.Portfolio import Portfolio
from src.application.BacktestEngine import BacktestEngine
from src.core.Performance import Performance

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'axes.facecolor': '#121212',
    'figure.facecolor': '#121212',
    'grid.color': '#2a2a2a',
    'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0',
    'lines.linewidth': 1.5,
    'font.family': 'sans-serif'
})

## 2. Environment and Component Instantiation
Below, we instantiate the domain aggregates and services, injecting them into the Engine to orchestrate the simulation in a decoupled manner. This adheres to rigorous Dependency Injection principles.

In [ ]:
# 1. Initialize the data handler pointing to the data/ directory
data_dir = os.path.join(project_root, 'data')
data_handler = ParquetDataHandler(dir_path=data_dir)

# 2. Instantiate the Strategy (SMA Crossover)
strategy = SMAStrategy(short_window=50, long_window=200)

# 3. Define the friction model (0.1% per transaction = 0.001)
friction_model = ProportionalFrictionModel(proportional_rate=0.001)

# 4. Create the Portfolio with initial capital (e.g., 100,000 USD)
initial_capital = 100000.0
portfolio = Portfolio(initial_capital=initial_capital, friction_model=friction_model)

# 5. Inject dependencies into the BacktestEngine
engine = BacktestEngine(data_handler=data_handler, strategy=strategy, portfolio=portfolio)

## 3. Vectorized Backtest Execution

In [ ]:
# Launch the backtest for the 'mock_data' dataset
# This triggers the asymptotically O(1) vectorized logic
symbol = 'mock_data'
engine.run_backtest(symbol)

# Extract the equity curve from the portfolio
equity_curve = portfolio.get_equity_curve()

# Load raw data to compute the 'Buy & Hold' baseline comparison
historical_data = data_handler.load_data(symbol)
bnh_returns = historical_data['close'].pct_change().fillna(0)
buy_and_hold_curve = initial_capital * (1.0 + bnh_returns).cumprod()

# Align time indices in case of shifts
equity_curve = equity_curve.reindex(historical_data.index).ffill().fillna(initial_capital)

## 4. Performance Metrics (`Performance.py`)

In [ ]:
# Vectorized calculation of metrics 
strategy_returns = equity_curve.pct_change().fillna(0)

sharpe = Performance.calculate_sharpe_ratio(strategy_returns, risk_free_rate=0.0)
mdd = Performance.calculate_max_drawdown(equity_curve)
cagr = Performance.calculate_cagr(equity_curve)
volatility = Performance.calculate_annualized_volatility(strategy_returns)

print("=== Performance Metrics ===")
print(f"Sharpe Ratio:       {sharpe:.2f}")
print(f"Maximum Drawdown:   {mdd:.2%}")
print(f"CAGR:               {cagr:.2%}")
print(f"Annual Volatility:  {volatility:.2%}")

## 5. Visualization

In [ ]:
fig, ax = plt.subplots()

ax.plot(equity_curve.index, equity_curve.values, label='SMA Strategy (Net of Friction)', color='#00d2ff', alpha=0.9)
ax.plot(buy_and_hold_curve.index, buy_and_hold_curve.values, label='Buy & Hold (Baseline)', color='#ff4b4b', alpha=0.6, linestyle='--')

ax.set_title('Portfolio Equity Curve: SMA Strategy vs Buy & Hold', fontsize=16, fontweight='bold', pad=20, color='white')
ax.set_ylabel('Capital (USD)', fontsize=12, labelpad=15)
ax.set_xlabel('Date', fontsize=12, labelpad=15)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)

ax.grid(True, linestyle=':', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#333333')
ax.spines['bottom'].set_color('#333333')

legend = ax.legend(loc='upper left', frameon=True, facecolor='#1e1e1e', edgecolor='#333333')
for text in legend.get_texts():
    text.set_color("white")

ax.fill_between(equity_curve.index, equity_curve.values, initial_capital, where=(equity_curve.values >= initial_capital), color='#00d2ff', alpha=0.1)
ax.fill_between(equity_curve.index, equity_curve.values, initial_capital, where=(equity_curve.values < initial_capital), color='#ff4b4b', alpha=0.1)

plt.tight_layout()
plt.show()